# EbbRAG — Data Preparation (SPEC-04)

Builds the corpora and query sets used across the experiment notebooks.

**Datasets:** Natural Questions (NQ-Open) · HotpotQA (distractor) · Wikipedia passages (DPR) · EbbRAG-Temporal (synthetic time-delta replication)

**Outputs:**
- `cache/wiki_chunks.pkl`, `cache/wiki_embeddings.npy` — Wikipedia chunk corpus + embeddings
- `data/nq_dev_linked.jsonl` — NQ-dev queries linked to relevant chunk ids
- `data/temporal/temporal_dataset.jsonl` — EbbRAG-Temporal dataset

## 1. Install & imports

In [ ]:
!pip install -q datasets>=2.19.0 sentence-transformers>=2.7.0 tqdm

import os, json, pickle, re
from pathlib import Path
import numpy as np
from datasets import load_dataset
from tqdm.auto import tqdm

os.makedirs('cache', exist_ok=True)
os.makedirs('data', exist_ok=True)
os.makedirs('data/temporal', exist_ok=True)

MAX_WIKI_DOCS = 50_000

print('Imports ✓')

## 2. Load NQ-Open and HotpotQA

In [ ]:
nq = load_dataset('nq_open', split='validation')
hotpot = load_dataset('hotpot_qa', 'distractor', split='validation')

print(f'NQ-Open (validation): {len(nq)} examples')
print(f'HotpotQA (distractor, validation): {len(hotpot)} examples')
print(nq[0])
print(hotpot[0]['question'], '->', hotpot[0]['answer'])

## 3. Build Wikipedia chunk corpus (DPR passages)

Capped at `MAX_WIKI_DOCS` for tractability. Cached to disk so this only runs once.

In [ ]:
WIKI_CHUNKS_PATH = 'cache/wiki_chunks.pkl'
WIKI_EMB_PATH = 'cache/wiki_embeddings.npy'

if os.path.exists(WIKI_CHUNKS_PATH):
    with open(WIKI_CHUNKS_PATH, 'rb') as f:
        wiki_chunks = pickle.load(f)
    print(f'Loaded {len(wiki_chunks)} cached wiki chunks ✓')
else:
    wiki = load_dataset('wiki_dpr', 'psgs_w100.nq.exact', split='train', streaming=True)
    wiki_chunks = []
    for i, row in enumerate(tqdm(wiki, total=MAX_WIKI_DOCS)):
        if i >= MAX_WIKI_DOCS:
            break
        wiki_chunks.append({'id': row['id'], 'text': row['text'], 'title': row.get('title', '')})
    with open(WIKI_CHUNKS_PATH, 'wb') as f:
        pickle.dump(wiki_chunks, f)
    print(f'Built and cached {len(wiki_chunks)} wiki chunks ✓')

In [ ]:
from sentence_transformers import SentenceTransformer

if os.path.exists(WIKI_EMB_PATH):
    wiki_embeddings = np.load(WIKI_EMB_PATH)
    print(f'Loaded cached embeddings: {wiki_embeddings.shape} ✓')
else:
    embed_model = SentenceTransformer('BAAI/bge-small-en-v1.5')
    texts = [c['text'] for c in wiki_chunks]
    wiki_embeddings = embed_model.encode(
        texts, batch_size=128, normalize_embeddings=True, show_progress_bar=True
    )
    np.save(WIKI_EMB_PATH, wiki_embeddings)
    print(f'Encoded and cached embeddings: {wiki_embeddings.shape} ✓')

## 4. Link NQ queries to relevant chunk ids

Uses substring matching of the gold short answer against chunk text as a weak-label heuristic.

In [ ]:
def link_query_to_chunks(answers, chunks, max_links=5):
    """Weak-label: a chunk is 'relevant' if it contains one of the gold answer strings."""
    linked = []
    for c in chunks:
        text_lower = c['text'].lower()
        if any(a.lower() in text_lower for a in answers):
            linked.append(c['id'])
        if len(linked) >= max_links:
            break
    return linked

nq_linked = []
for ex in tqdm(nq):
    chunk_ids = link_query_to_chunks(ex['answer'], wiki_chunks)
    nq_linked.append({
        'query_id': f"nq-{len(nq_linked)}",
        'question': ex['question'],
        'answers': ex['answer'],
        'relevant_chunk_ids': chunk_ids,
    })

with open('data/nq_dev_linked.jsonl', 'w') as f:
    for row in nq_linked:
        f.write(json.dumps(row) + '\n')

linked_count = sum(1 for r in nq_linked if r['relevant_chunk_ids'])
print(f'Linked {linked_count}/{len(nq_linked)} NQ queries to at least one chunk ✓')

## 5. Build EbbRAG-Temporal dataset

Replicates each linked query across 4 time deltas (0, 7, 30, 90 days) **without updating the index between timesteps** — this isolates the effect of stability decay/reinforcement on retrieval quality over time.

In [ ]:
TIME_DELTAS_DAYS = [0, 7, 30, 90]
SECONDS_PER_DAY = 86400

temporal_rows = []
for row in nq_linked:
    if not row['relevant_chunk_ids']:
        continue
    for delta_days in TIME_DELTAS_DAYS:
        temporal_rows.append({
            'query_id': f"{row['query_id']}-t{delta_days}",
            'base_query_id': row['query_id'],
            'question': row['question'],
            'answers': row['answers'],
            'relevant_chunk_ids': row['relevant_chunk_ids'],
            'delta_days': delta_days,
            't_offset_seconds': delta_days * SECONDS_PER_DAY,
        })

with open('data/temporal/temporal_dataset.jsonl', 'w') as f:
    for row in temporal_rows:
        f.write(json.dumps(row) + '\n')

print(f'Built EbbRAG-Temporal dataset: {len(temporal_rows)} rows '
      f'({len(temporal_rows) // len(TIME_DELTAS_DAYS)} base queries × {len(TIME_DELTAS_DAYS)} deltas) ✓')

## 6. Sanity checks

In [ ]:
assert len(wiki_chunks) == wiki_embeddings.shape[0], 'chunk/embedding count mismatch'
assert all(r['delta_days'] in TIME_DELTAS_DAYS for r in temporal_rows)

print('Wiki chunks:        ', len(wiki_chunks))
print('Wiki embeddings:    ', wiki_embeddings.shape)
print('NQ linked queries:  ', len(nq_linked))
print('Temporal dataset:   ', len(temporal_rows))
print('Data prep complete ✓')